# Mantis-8M features on LIAS

Classifies the LIAS motor recordings (class 1, 2 or 3) using zero-shot features from the pretrained
Mantis-8M model, then trains SVM and logistic regression on them. Same steps as the MOMENT and TSFresh notebooks.

In [ ]:
import sys
from pathlib import Path

# Add the repo root so LIAS/ and common/ can be imported
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
import pandas as pd
import torch
from sklearn.metrics import accuracy_score
from mantis.architecture import MantisV1
from mantis.trainer import MantisTrainer

from LIAS.prepare_LIAS import process_all_LIAS_files, add_labels_LIAS
from common.mantis_embeddings import get_embeddings_in_batches, embeddings_to_dataframe
from common.feature_merging import rename_and_merge
from common.classification import (train_test_split_LIAS, scale_features,
                                   train_svc_model, train_log_reg_model, evaluate_model)

## Load Mantis-8M

In [ ]:
# Use the GPU if there is one (NVIDIA or Apple), otherwise the CPU
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("Using device:", device)

# Pretrained Mantis-8M, used as is (zero-shot): no fine-tuning on LIAS
network = MantisV1(device=device).from_pretrained("paris-noah/Mantis-8M")
model = MantisTrainer(device=device, network=network)

## Window the data and extract features

In [ ]:
# Windows of 1024 samples with 50% overlap. Each window is downsampled to 512 samples,
# which suits Mantis (it needs a length that is a multiple of 32).
WINDOW_SIZE = 1024
OVERLAP = int(WINDOW_SIZE/2)

# One list of windows per phase: a, b and c
all_processed_windows_a, all_processed_windows_b, all_processed_windows_c = process_all_LIAS_files(WINDOW_SIZE, OVERLAP)

print(f"Total processed windows for 'isa': {len(all_processed_windows_a)}")
print(f"Total processed windows for 'isb': {len(all_processed_windows_b)}")
print(f"Total processed windows for 'isc': {len(all_processed_windows_c)}")

In [ ]:
# Mantis gives 256 features per window, for each phase
batch_size = 256  # lower this if the GPU runs out of memory
combined_embeddings_a, combined_metadata_a = get_embeddings_in_batches(all_processed_windows_a, model, batch_size=batch_size)
combined_embeddings_b, combined_metadata_b = get_embeddings_in_batches(all_processed_windows_b, model, batch_size=batch_size)
combined_embeddings_c, combined_metadata_c = get_embeddings_in_batches(all_processed_windows_c, model, batch_size=batch_size)

In [ ]:
# One table per phase: file_name, window_num and the 256 features
df_embeddings_a = embeddings_to_dataframe(combined_embeddings_a, combined_metadata_a)
df_embeddings_b = embeddings_to_dataframe(combined_embeddings_b, combined_metadata_b)
df_embeddings_c = embeddings_to_dataframe(combined_embeddings_c, combined_metadata_c)

In [ ]:
# Combine the three phases into one table (one row per window).
# Feature columns are renamed to mantis_<phase>_<number>, e.g. mantis_a_17
df_merged = rename_and_merge(df_embeddings_a, df_embeddings_b, df_embeddings_c, feature_type="mantis")

print("Merged DataFrame head:")
display(df_merged.head())

# file_name, window_num + 256 * 3 = 768 features, so 770 columns
print(f"Number of columns in merged dataframe: {df_merged.shape[1]}")

## Train-test split (by file)

In [ ]:
# Split by file, not by window: windows from the same recording are very alike, so if a file
# had windows on both sides, the test set would contain near-copies of training data.
# add_labels_LIAS keeps the 132 labelled files and adds the 'class' and 'file_group' columns.
# The split is stratified on file_group (condition and speed). Change the seed for a different split.
df_labelled = add_labels_LIAS(df_merged)
train_df, test_df = train_test_split_LIAS(df_labelled, test_size=0.2, seed=67)

In [ ]:
# No file should be in both sets: this should give an empty set
set(train_df["file_name"]) & set(test_df["file_name"])

In [ ]:
# Each file_group should have about the same share in both sets
print(train_df["file_group"].value_counts(normalize=True))
print(test_df["file_group"].value_counts(normalize=True))

## Scaling

In [ ]:
# Scaled features: each feature gets mean 0 and standard deviation 1.
# The scaler is fit on the training set only, so nothing about the test set leaks into training.
X_train_scaled, y_train, X_test_scaled, y_test = scale_features(train_df, test_df, feature_type="mantis")

# Unscaled features, to compare against
X_train = train_df.filter(regex="^mantis_")
X_test = test_df.filter(regex="^mantis_")

print(f"X_train_scaled: {X_train_scaled.shape}, X_test_scaled: {X_test_scaled.shape}")

## Models and evaluation

Each model is trained on unscaled and on scaled features. The RBF SVM sets the width of its kernel
from the spread of all feature values together. The unscaled Mantis features sit at very different
levels (average size from about 0.04 to 13), which makes the kernel far too wide, so the model can
hardly tell the classes apart. Scaling puts every feature on the same footing.

In [ ]:
# SVM with RBF kernel on unscaled features
svm_model_unscaled_rbf = train_svc_model(X_train, y_train, kernel='rbf')
y_pred_unscaled_rbf = svm_model_unscaled_rbf.predict(X_test)

evaluate_model(y_test, y_pred_unscaled_rbf, "RBF Kernel SVM on Unscaled Data")

In [ ]:
# SVM with linear kernel on unscaled features
svm_model_unscaled_linear = train_svc_model(X_train, y_train, kernel='linear')
y_pred_unscaled_linear = svm_model_unscaled_linear.predict(X_test)

evaluate_model(y_test, y_pred_unscaled_linear, "Linear Kernel SVM on Unscaled Data")

In [ ]:
# SVM with RBF kernel on scaled features
svm_model_scaled_rbf = train_svc_model(X_train_scaled, y_train, kernel='rbf')
y_pred_scaled_rbf = svm_model_scaled_rbf.predict(X_test_scaled)

evaluate_model(y_test, y_pred_scaled_rbf, "RBF Kernel SVM on Scaled Data")

In [ ]:
# SVM with linear kernel on scaled features
svm_model_scaled_linear = train_svc_model(X_train_scaled, y_train, kernel='linear')
y_pred_scaled_linear = svm_model_scaled_linear.predict(X_test_scaled)

evaluate_model(y_test, y_pred_scaled_linear, "Linear Kernel SVM on Scaled Data")

In [ ]:
# Logistic regression on unscaled features
log_reg_model_unscaled = train_log_reg_model(X_train, y_train)
y_pred_log_reg_unscaled = log_reg_model_unscaled.predict(X_test)

evaluate_model(y_test, y_pred_log_reg_unscaled, "Logistic Regression on Unscaled Data")

In [ ]:
# Logistic regression on scaled features
log_reg_model_scaled = train_log_reg_model(X_train_scaled, y_train)
y_pred_log_reg_scaled = log_reg_model_scaled.predict(X_test_scaled)

evaluate_model(y_test, y_pred_log_reg_scaled, "Logistic Regression on Scaled Data")

## Summary

In [ ]:
# Test accuracy of every model, unscaled vs scaled
predictions = {
    "RBF SVM":             (y_pred_unscaled_rbf,     y_pred_scaled_rbf),
    "Linear SVM":          (y_pred_unscaled_linear,  y_pred_scaled_linear),
    "Logistic regression": (y_pred_log_reg_unscaled, y_pred_log_reg_scaled),
}

summary = pd.DataFrame(
    {name: {"Unscaled": accuracy_score(y_test, unscaled), "Scaled": accuracy_score(y_test, scaled)}
     for name, (unscaled, scaled) in predictions.items()}
).T.round(2)
summary